# Colab / Kaggle runner (Task 12, GPU steps)
Shell cells only: all logic lives in `src/`. Select a **T4 GPU** runtime first (Runtime > Change runtime type).
Members may split the 8 runs across accounts; each commits its own config or NOTES changes.

**Dependencies.** Run the install cell below. If `pip` cannot find `torch==2.14.1` / `torchvision==0.29.1` for the Colab Python, do **not** fight it: keep Colab's preinstalled torch/torchvision, and install the rest with
`grep -v -E '^(torch|torchvision)==' requirements.txt > /tmp/req.txt && pip install -r /tmp/req.txt`.
Note the torch version you ended up with in `outputs/NOTES.md` so the run stays reproducible.

In [ ]:
!nvidia-smi
!git clone https://github.com/hla06/From-Pixels-to-Language
%cd From-Pixels-to-Language

In [ ]:
!pip install -r requirements.txt

In [ ]:
# Fallback if the pinned torch is unavailable (see the note above):
!grep -v -E '^(torch|torchvision)==' requirements.txt > /tmp/req.txt && pip install -r /tmp/req.txt

In [ ]:
!wandb login

In [ ]:
# Colab resets wipe local disk, so every result folder lives on Drive and is symlinked into the repo.
# Re-run this cell first in every new session; training cells use --resume to skip finished runs.
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/pixels-to-language/final
!mkdir -p outputs
for d in checkpoints history eval tuning figures eda:
    !mkdir -p /content/drive/MyDrive/pixels-to-language/final/{d}
    !rm -rf outputs/{d} && ln -s /content/drive/MyDrive/pixels-to-language/final/{d} outputs/{d}
!ls -la outputs

## Recover trained checkpoints from W&B (after a runtime reset)
Every finished run uploaded its best checkpoint as the W&B artifact `<run>-best`. Run this instead of retraining, then jump to **Step 5**. Needs `wandb login` first.

In [ ]:
!python -m src.cli fetch-checkpoints

Rebuild the per-epoch training curves of every run from W&B (the history files are lost on a reset). Run `figures` afterwards to redraw the curves and the validation-vs-test table.

In [ ]:
!python -m src.cli fetch-history

## Step 2: EDA and timing

In [ ]:
!python -m src.cli eda --config configs/scratch_cnn.yaml

Time one ResNet-18 fine-tune epoch (`epochs: 1` override is not available from the CLI, so edit `epochs: 1` in a scratch copy of the config). Use seconds/epoch to set final `epochs` and `patience` so all runs fit; record the number in `outputs/NOTES.md`.

In [ ]:
!sed 's/^  epochs: .*/  epochs: 1/; s/run_name: .*/run_name: timing/; s/name: wandb/name: "null"/' configs/resnet18_finetune.yaml > /tmp/timing.yaml
!echo "history_dir: outputs/smoke" >> /tmp/timing.yaml
!cat /tmp/timing.yaml
!time python -m src.cli train --task classification --config /tmp/timing.yaml

## Step 3: the 8 training runs (tracker group `m1-runs`)
Run after setting final epochs/patience in the configs. One cell per run.

In [ ]:
!python -m src.cli train --task classification --config configs/scratch_cnn.yaml --resume

In [ ]:
!python -m src.cli train --task classification --config configs/resnet18_random.yaml --resume

In [ ]:
!python -m src.cli train --task classification --config configs/resnet18_frozen.yaml --resume

In [ ]:
!python -m src.cli train --task classification --config configs/resnet18_finetune.yaml --resume

In [ ]:
!python -m src.cli train --task classification --config configs/resnet50_frozen.yaml --resume

In [ ]:
!python -m src.cli train --task classification --config configs/resnet50_finetune.yaml --resume

In [ ]:
!python -m src.cli train --task classification --config configs/efficientnet_b0_frozen.yaml --resume

In [ ]:
!python -m src.cli train --task classification --config configs/efficientnet_b0_finetune.yaml --resume

## Step 4: hyperparameter tuning (Grid vs Random on ResNet-18)

In [ ]:
!python -m src.cli tune --method grid --config configs/tune_resnet18.yaml --resume

In [ ]:
!python -m src.cli tune --method random --config configs/tune_resnet18.yaml --resume

Retrain the best setting once at the full budget **without overwriting `resnet18_finetune`**: edit `configs/resnet18_tuned.yaml` and copy from `outputs/tuning/<method>.json` (`best_params`): `lr_backbone` = best lr, `lr_head` = 10 x best lr, plus the best `weight_decay` and `dropout`. Then run the cell below.

In [ ]:
!python -m src.cli train --task classification --config configs/resnet18_tuned.yaml --resume

## Step 5: evaluate, Grad-CAM, figures

In [ ]:
!python -m src.cli evaluate --checkpoint outputs/checkpoints/scratch_cnn.pt --name scratch_cnn

In [ ]:
!python -m src.cli evaluate --checkpoint outputs/checkpoints/resnet18_random.pt --name resnet18_random

In [ ]:
!python -m src.cli evaluate --checkpoint outputs/checkpoints/resnet18_frozen.pt --name resnet18_frozen

In [ ]:
!python -m src.cli evaluate --checkpoint outputs/checkpoints/resnet18_finetune.pt --name resnet18_finetune

In [ ]:
!python -m src.cli evaluate --checkpoint outputs/checkpoints/resnet18_tuned.pt --name resnet18_tuned

In [ ]:
!python -m src.cli evaluate --checkpoint outputs/checkpoints/resnet50_frozen.pt --name resnet50_frozen

In [ ]:
!python -m src.cli evaluate --checkpoint outputs/checkpoints/resnet50_finetune.pt --name resnet50_finetune

In [ ]:
!python -m src.cli evaluate --checkpoint outputs/checkpoints/efficientnet_b0_frozen.pt --name efficientnet_b0_frozen

In [ ]:
!python -m src.cli evaluate --checkpoint outputs/checkpoints/efficientnet_b0_finetune.pt --name efficientnet_b0_finetune

Grad-CAM: swap the checkpoint (and output folder) in the next cell to the **winning model** (e.g. `resnet18_tuned.pt` if it beat `resnet18_finetune`).

In [ ]:
!python -m src.cli gradcam --checkpoint outputs/checkpoints/resnet50_finetune.pt --out outputs/figures/gradcam_resnet50_finetune

In [ ]:
!python -m src.cli gradcam --checkpoint outputs/checkpoints/scratch_cnn.pt --out outputs/figures/gradcam_scratch_cnn

In [ ]:
!python -m src.cli figures

## Copy results to Drive

In [ ]:
# Results already live on Drive (outputs/* are symlinks). Nothing to copy.
!ls /content/drive/MyDrive/pixels-to-language/final/checkpoints